# SROIE Dataset Initial Exploration

This notebook loads the local SROIE dataset through `src.data.sroie` and performs initial validation and exploratory analysis. The raw dataset must be available at `data/raw/SROIE/` or supplied through `DATASET_ROOT`.

In [2]:
from collections import Counter
from pathlib import Path
import sys

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.sroie import SROIEValidationError, load_sroie_dataset

DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "SROIE"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

In [3]:
try:
    dataset = load_sroie_dataset(DATASET_ROOT)
    validation_issues = []
except SROIEValidationError as error:
    validation_issues = list(error.issues)
    print(f"Validation issues: {len(validation_issues)}")
    for issue in validation_issues[:20]:
        print(f"- {issue}")
    if len(validation_issues) > 20:
        print(f"... and {len(validation_issues) - 20} more")
    dataset = load_sroie_dataset(DATASET_ROOT, strict=False)

print(f"Total valid samples: {len(dataset)}")
print(f"Total validation issues: {len(validation_issues)}")
print("Issue counts:")
print(pd.Series(Counter(issue.kind for issue in validation_issues), name="count"))
dataset.head()

FileNotFoundError: SROIE dataset root does not exist: c:\Users\chech\Documents\AI-Document-Intelligence\data\raw\SROIE

In [ ]:
def image_dimensions(image_path: str) -> tuple[int | None, int | None]:
    image = cv2.imread(image_path)
    if image is None:
        return None, None
    height, width = image.shape[:2]
    return width, height

if not dataset.empty:
    dimensions = dataset["image_path"].map(image_dimensions)
    dataset[["width", "height"]] = pd.DataFrame(dimensions.tolist(), index=dataset.index)
    dataset["aspect_ratio"] = dataset["width"] / dataset["height"]
else:
    dataset["width"] = pd.Series(dtype="float64")
    dataset["height"] = pd.Series(dtype="float64")
    dataset["aspect_ratio"] = pd.Series(dtype="float64")

dataset[["width", "height", "aspect_ratio"]].describe()

In [ ]:
text_fields = ["company", "date", "address", "total"]
text_lengths = dataset[text_fields].apply(lambda column: column.str.len())
text_lengths.describe().T.rename(columns={"count": "samples"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(dataset["aspect_ratio"].dropna(), bins=30, ax=axes[0])
axes[0].set_title("Image aspect ratios")
axes[0].set_xlabel("width / height")

sns.histplot(text_lengths["address"], bins=30, ax=axes[1])
axes[1].set_title("Address text lengths")
axes[1].set_xlabel("characters")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for axis, field in zip(axes.flat, text_fields):
    sns.histplot(text_lengths[field], bins=20, ax=axis)
    axis.set_title(f"{field} length")
    axis.set_xlabel("characters")
plt.tight_layout()
plt.show()

In [ ]:
sample_count = min(6, len(dataset))
if sample_count:
    figure, axes = plt.subplots(2, 3, figsize=(18, 11))
    axes = axes.ravel()
    for axis, (_, row) in zip(axes, dataset.head(sample_count).iterrows()):
        image = cv2.cvtColor(cv2.imread(row["image_path"]), cv2.COLOR_BGR2RGB)
        axis.imshow(image)
        axis.axis("off")
        fields = "\n".join(f"{field}: {row[field]}" for field in text_fields)
        axis.set_title(f"{row['image_id']}\n{fields}", fontsize=9, loc="left")
    for axis in axes[sample_count:]:
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No valid receipt images are available to display.")

In [ ]:
metadata_columns = ["image_id", "image_path", *text_fields]
metadata_path = PROCESSED_ROOT / "sroie_metadata.csv"
dataset[metadata_columns].to_csv(metadata_path, index=False)
print(f"Saved lightweight metadata for {len(dataset)} valid samples to {metadata_path}")